# 訓練

In [ ]:
# 1. 安裝 Hugging Face 核心黃金四件套
print("📦 正在安裝 Hugging Face 官方微調依賴套件...")
!pip install -q transformers peft bitsandbytes datasets accelerate trl

In [ ]:
import json
import torch
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, TaskType, get_peft_model

# 2. 智慧讀取並融合異質資料集
print("\n🗂️ 正在讀取並融合異質網路安全資料集...")
combined_data = []

try:
    with open("/kaggle/input/datasets/alice0107/llmsec/hdfs_matrix_train.json", "r", encoding="utf-8") as f: # Change the file location
        combined_data.extend(json.load(f))
except:
    print("⚠️ 提示：未找到日誌。")


# 3. 配置 Qwen2.5-3B 專用 Tokenizer 與 4-bit 規格
model_id = "Qwen/Qwen2.5-3B-Instruct"
print(f"\n📥 正在從 Hugging Face 下載 {model_id} 骨幹與配置...")

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("📝 正在進行資料集高效 Tokenize 預處理...")
def tokenize_function(example):
    full_text = f"<|im_start|>system\n{example['instruction']}<|im_end|>\n<|im_start|>user\n{example['input']}<|im_end|>\n<|im_start|>assistant\n{example['output']}<|im_end|>"
    return tokenizer(full_text, truncation=True, max_length=1024)

raw_dataset = Dataset.from_list(combined_data)
hf_dataset = raw_dataset.map(tokenize_function, remove_columns=raw_dataset.column_names)
print(f"✅ 資料 Tokenize 融合成功！總訓練樣本數: {len(hf_dataset)} 筆。")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# 【核心防禦配置】：強迫模型開啟梯度檢查點的底層支援，以在 T4 上最大化節省 VRAM
model.gradient_checkpointing_enable()

# 4. 配置高強度資安 LoRA 參數并手動注入基礎模型
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# 5. 配置專為免費版 T4 降溫優化的 TrainingArguments
output_dir = "/kaggle/working/qwen2_5_3b_security_lora"
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,     
    gradient_accumulation_steps=8,     
    gradient_checkpointing=True,        
    optim="paged_adamw_8bit",         
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    logging_steps=5,
    num_train_epochs=3.0,
    fp16=True,                      
    save_strategy="no",
    report_to="none"
)

# 6. 宣告標準語言模型 Collator
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 7. 啟動官方正統 Trainer 引擎
trainer = Trainer(
    model=model,
    train_dataset=hf_dataset,
    data_collator=data_collator,
    args=training_args
)

print("\n🔥 官方原生微調引擎啟動！正在跨維度對齊資安專家 JSON 語意...")
train_result = trainer.train()

# =====================================================================
# 📊 自動化 Loss 曲線繪圖與審計工具
# =====================================================================
print("\n📈 正在自適應擷取訓練歷史，生成安全核心 Loss 曲線...")

log_history = trainer.state.log_history
steps = [log["step"] for log in log_history if "loss" in log]
losses = [log["loss"] for log in log_history if "loss" in log]

if steps and losses:
    plt.figure(figsize=(10, 5))
    plt.plot(steps, losses, label="Training Loss", color="#1f77b4", linewidth=2.5, marker='o', markersize=4)
    plt.title("Qwen2.5-3B Multi-Task Security Fine-Tuning Loss Curve", fontsize=12, fontweight='bold')
    plt.xlabel("Training Steps", fontsize=10)
    plt.ylabel("Cross-Entropy Loss", fontsize=10)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend(fontsize=10)

    plt.savefig("/kaggle/working/security_fine_tuning_loss.png", dpi=300)
    plt.show()
    print("💾 完美的 Loss 收斂曲線圖已儲存至: /kaggle/working/security_fine_tuning_loss.png")
else:
    print("⚠️ 提示：訓練步數過短，未能成功擷取到足夠的 Loss 歷史紀錄。")

# 8. 定向導出並儲存輕量化 LoRA 權重向量 (Vector)
print("\n💾 正在精煉並儲存 20MB 的資安專家 LoRA 權重向量...")
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

In [ ]:
# 9. 自動封裝下載 ZIP 檔
import shutil
shutil.make_archive("/kaggle/working/Qwen2_5_3B_HF_Native_LoRA.zip", 'zip', output_dir)
print("🎉 微調適配圓滿成功！已自動調用瀏覽器下載最終 LoRA Vector。")

# 測試

In [ ]:
# 1. 【黃金核心】：直接把卡住 PEFT 框架的 torchao 強制升級至最新版！
print("📦 正在暴力升級 torchao 加速庫以破除 PEFT 版本封鎖...")
!pip install -q -U torchao

In [ ]:
# =====================================================================
# 🌐 【系統偵測專用】HDFS 多分類安全邊界壓力測試
# =====================================================================
import torch
import json
from transformers import AutoTokenizer
from peft import AutoPeftModelForCausalLM

lora_path = "/kaggle/working/qwen2_5_3b_security_lora"

print("⚙️ 正在使用官方 AutoPeft 引擎，一鍵乾淨載入 4-bit 基座與專屬 LoRA 靈魂...")
tokenizer = AutoTokenizer.from_pretrained(lora_path)

# 智慧一鍵載入：這會自動去讀取 lora_path 裡的配置，自動抓基座模型並完成融合，永不打結！
covert_model = AutoPeftModelForCausalLM.from_pretrained(
    lora_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

# 【測試輸入點】：給模型餵入那筆最有價值的「INFO 級別隱蔽型異常日誌」
test_prompt = """您是一位部署於 Next-Gen SOC Operations 的分散式系統審計 Copilot。請檢視以下大型雲端檔案系統（HDFS）產生的原始系統日誌（Raw Logs），深度結合日誌級別與行為上下文，判斷該系統行為是否屬於異常攻擊或配置故障。請將分析結果以嚴格的無損 JSON 格式返回，嚴禁包含任何額外的 Markdown 標籤或文字解釋。
[HDFS Runtime Telemetry]\n- Timestamp: 081109 203530\n- PID: 179 | Log Level: INFO\n- Component: dfs.DataNode$DataXceiver\n- Block ID: blk_-3544583377289625738\n- Raw Log Content: 10.251.197.226:50010 Served block blk_-3544583377289625738 to /10.251.75.79"""

# 拼裝對齊訓練時的 Qwen2.5 專屬 Chat 邊界模板
formatted_input = f"<|im_start|>system\n您是一位部署於 Next-Gen SOC Operations 的分散式系統審計 Copilot。請檢視以下大型雲端檔案系統（HDFS）產生的原始系統日誌（Raw Logs），深度結合日誌級別與行為上下文，判斷該系統行為是否屬於異常攻擊或配置故障。請將分析結果以嚴格的無損 JSON 格式返回，嚴禁包含任何額外的 Markdown 標籤或文字解釋。<|im_end|>\n<|im_start|>user\n{test_prompt}<|im_end|>\n<|im_start|>assistant\n"

inputs = tokenizer(formatted_input, return_tensors="pt").to("cuda")

print("\n🤖 核心安全通感甦醒！模型正在進行脈絡行為鏈推理...")
with torch.no_grad():
    outputs = covert_model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=False)

# 解碼並精準切取出 Assistant 的專家回覆
result_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
response_only = result_text.split("<|im_start|>assistant\n")[-1].split("<|im_end|>")[0].strip()

print("\n🔥 【真・資安專家模型】四象限矩陣解耦 - 專家級 JSON 審計輸出：")
print(response_only)